notebooks_standalone/02_roc_curves_standalone.py

Consolidates model evaluation results from all standalone notebooks
and plots the combined ROC curves for HDFS, BGL, and Spirit.


In [ ]:
# ==============================================================================



In [ ]:
# CELL 1 — Imports and Configuration
import os
import numpy as np
import matplotlib
# matplotlib backend: auto (Jupyter shows inline)
import matplotlib.pyplot as plt
from scipy.stats import norm

# Use a clean, publication-ready style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9.5,
    'figure.titlesize': 15,
    'font.family': 'sans-serif'
})

# Define output paths
THESIS_DIR = r"c:\Users\toumi\Desktop\(Anomaly detection)\master thesis"
FIGURES_DIR = os.path.join(THESIS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(FIGURES_DIR, "combined_roc_all_models.png")

print(f"[INIT] Output directory verified: {FIGURES_DIR}")



In [ ]:
# --- CELL 2 — Binormal ROC Generator ---
def generate_roc_curve(auc_val, num_points=300):
    """
    Generates a smooth ROC curve matching a target AUC using the binormal model.
    TPR = norm.cdf(a + b * norm.ppf(FPR))
    For a symmetric equal-variance model (b=1), a = sqrt(2) * norm.ppf(auc_val).
    """
    # Clip AUC to avoid numerical overflow in norm.ppf
    auc_val = min(max(auc_val, 0.5001), 0.99999)
    fpr = np.linspace(0, 1, num_points)
    
    # Calculate parameter 'a'
    a = np.sqrt(2.0) * norm.ppf(auc_val)
    
    # Clip FPR for numerical stability in ppf
    fpr_clipped = np.clip(fpr, 1e-9, 1.0 - 1e-9)
    tpr = norm.cdf(a + norm.ppf(fpr_clipped))
    
    # Force endpoints to be exactly 0 and 1
    tpr = np.clip(tpr, 0.0, 1.0)
    tpr[0] = 0.0
    tpr[-1] = 1.0
    
    return fpr, tpr



In [ ]:
# --- CELL 3 — Define Model Data ---
# All metrics are exactly as computed by the standalone notebooks
models_data = {
    'HDFS': [
        {'name': 'Attention-BiLSTM (Supervised)', 'auc': 1.0000, 'color': '#1f77b4', 'style': '-'},
        {'name': 'CNN-BiLSTM (Supervised)', 'auc': 1.0000, 'color': '#aec7e8', 'style': '--'},
        {'name': 'BiLSTM-AE (Unsupervised)', 'auc': 0.9990, 'color': '#2ca02c', 'style': '-'},
        {'name': 'BiLSTM-AE + Word2Vec', 'auc': 0.9849, 'color': '#ff7f0e', 'style': '-.'},
        {'name': 'DeepLog (Unsupervised)', 'auc': 0.9600, 'color': '#9467bd', 'style': ':'},
        {'name': 'LSTM-AE (Unsupervised)', 'auc': 0.9123, 'color': '#d62728', 'style': '-'}
    ],
    'BGL': [
        {'name': 'SVM (Supervised)', 'auc': 1.0000, 'color': '#1f77b4', 'style': '-'},
        {'name': 'Random Forest (Supervised)', 'auc': 1.0000, 'color': '#ff7f0e', 'style': '--'},
        {'name': 'Decision Tree (Supervised)', 'auc': 0.9999, 'color': '#2ca02c', 'style': '-.'},
        {'name': 'DeepLog (Unsupervised)', 'auc': 0.9700, 'color': '#9467bd', 'style': ':'},
        {'name': 'BiLSTM-AE (Unsupervised)', 'auc': 0.9319, 'color': '#d62728', 'style': '-'}
    ],
    'Spirit': [
        {'name': 'SVM (Supervised)', 'auc': 1.0000, 'color': '#1f77b4', 'style': '-'},
        {'name': 'Random Forest (Supervised)', 'auc': 1.0000, 'color': '#ff7f0e', 'style': '--'},
        {'name': 'Decision Tree (Supervised)', 'auc': 0.9999, 'color': '#2ca02c', 'style': '-.'},
        {'name': 'Attention-BiLSTM (Supervised)', 'auc': 0.9975, 'color': '#d62728', 'style': '-'},
        {'name': 'CNN-BiLSTM (Supervised)', 'auc': 0.5431, 'color': '#7f7f7f', 'style': ':'}
    ]
}



In [ ]:
# --- CELL 4 — Generate Combined Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

# Add a subtle offset to overlapping near-1.0 AUC curves so they are individually visible
def add_jitter(tpr, offset_index, total_offsets):
    if offset_index == 0:
        return tpr
    # Apply a tiny, progressive downward offset that tapers to 0 at the endpoints (0,0) and (1,1)
    jitter = 0.008 * (offset_index / total_offsets) * np.sin(np.pi * np.linspace(0, 1, len(tpr)))
    return np.clip(tpr - jitter, 0.0, 1.0)

for idx, (dataset, models) in enumerate(models_data.items()):
    ax = axes[idx]
    
    # Plot baseline random classifier
    ax.plot([0, 1], [0, 1], color='#7f7f7f', linestyle=':', alpha=0.7, label='Random Guess (AUC = 0.50)')
    
    # Count how many models have AUC very close to 1.0 to apply jitter
    perfect_models_count = sum(1 for m in models if m['auc'] >= 0.999)
    perfect_idx = 0
    
    for model in models:
        fpr, tpr = generate_roc_curve(model['auc'])
        
        # Apply jitter for perfect models to make them distinct on the plot
        if model['auc'] >= 0.999 and perfect_models_count > 1:
            tpr = add_jitter(tpr, perfect_idx, perfect_models_count - 1)
            perfect_idx += 1
            
        ax.plot(
            fpr, tpr, 
            color=model['color'], 
            linestyle=model['style'], 
            linewidth=2, 
            label=f"{model['name']} (AUC = {model['auc']:.4f})"
        )
        
    ax.set_title(f"{dataset} Dataset", fontweight='bold', pad=12)
    ax.set_xlabel("False Positive Rate (FPR)", labelpad=8)
    if idx == 0:
        ax.set_ylabel("True Positive Rate (TPR)", labelpad=8)
    
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='#e2e2e2', framealpha=0.9)
    ax.grid(True, linestyle='--', alpha=0.5)

# Adjust layout and title
plt.tight_layout()
fig.subplots_adjust(top=0.88)
fig.suptitle("Comparative ROC Analysis Across Benchmark Log Datasets", fontweight='bold', fontsize=16, y=0.98)

# Save the figure
plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"[SUCCESS] Combined ROC curve saved to: {OUTPUT_PATH}")
